In [2]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

In [10]:
df = pd.read_excel(r"D:\Learnerea\video content creation\creditrisk_with_pandas\raw data\synthetic_credit_accounts_all_buckets - v2.xlsx")
df.sample(5)

,customer_id,customer_name,account_id,outstanding_balance,bill_generation_date,due_date,payment_date,snapshot_date
98,CUST90129,Yuzvendra C,ACC90129,48900.00,2026-06-20,2026-07-10,NaT,2026-09-30
10,CUST1018,Rohan Patel,ACC20018,96941.34,2026-08-24,2026-09-13,2026-09-16,2026-09-30
49,CUST1077,Rahul Gupta,ACC20077,403508.79,2026-09-03,2026-09-21,2026-09-28,2026-09-30
73,CUST90104,Kiara Advani,ACC90104,18500.25,2026-09-05,2026-09-25,NaT,2026-09-30
91,CUST90122,Rohit Sharma,ACC90122,125400.90,2026-07-26,2026-08-15,NaT,2026-09-30


In [11]:
indices = [79, 102, 87, 7, 12]
sample_df = df.iloc[indices]
sample_df

,customer_id,customer_name,account_id,outstanding_balance,bill_generation_date,due_date,payment_date,snapshot_date
79,CUST90110,Siddharth Roy,ACC90110,134700.00,2026-08-19,2026-09-08,NaT,2026-09-30
102,CUST90133,Sachin T,ACC90133,3250000.65,2026-05-20,2026-06-09,NaT,2026-09-30
87,CUST90118,Deepika P,ACC90118,3210400.00,2026-07-18,2026-08-07,NaT,2026-09-30
7,CUST1014,Kavya Joshi,ACC20014,172396.50,2026-07-14,2026-08-04,2026-08-21,2026-09-30
12,CUST1021,Isha Singh,ACC20021,142997.02,2026-06-27,2026-07-18,2026-08-27,2026-09-30


In [17]:
df['final_payment_date'] = df['payment_date'].combine_first(df['snapshot_date'])
df.head()

,customer_id,customer_name,account_id,outstanding_balance,bill_generation_date,due_date,payment_date,snapshot_date,final_payment_date
0,CUST1003,Ishaan Mehta,ACC20099,102016.73,2026-06-29,2026-07-19,2026-07-02,2026-09-30,2026-07-02
1,CUST1004,Radhika Kapoor,ACC20004,178231.99,2026-07-30,2026-08-19,2026-08-15,2026-09-30,2026-08-15
2,CUST1005,Aarav Gupta,ACC20005,2092078.40,2026-06-17,2026-07-08,2026-06-20,2026-09-30,2026-06-20
3,CUST1007,Manish Mehta,ACC20007,422769.57,2026-06-30,2026-07-21,2026-07-04,2026-09-30,2026-07-04
4,CUST1008,Meera Gupta,ACC20008,43325.90,2026-06-18,2026-07-03,2026-06-25,2026-09-30,2026-06-25


In [19]:
df['dpd'] = ((df['final_payment_date'] - df['due_date']).dt.days).clip(lower=0)
df.head()

,customer_id,customer_name,account_id,outstanding_balance,bill_generation_date,due_date,payment_date,snapshot_date,final_payment_date,dpd
0,CUST1003,Ishaan Mehta,ACC20099,102016.73,2026-06-29,2026-07-19,2026-07-02,2026-09-30,2026-07-02,0
1,CUST1004,Radhika Kapoor,ACC20004,178231.99,2026-07-30,2026-08-19,2026-08-15,2026-09-30,2026-08-15,0
2,CUST1005,Aarav Gupta,ACC20005,2092078.40,2026-06-17,2026-07-08,2026-06-20,2026-09-30,2026-06-20,0
3,CUST1007,Manish Mehta,ACC20007,422769.57,2026-06-30,2026-07-21,2026-07-04,2026-09-30,2026-07-04,0
4,CUST1008,Meera Gupta,ACC20008,43325.90,2026-06-18,2026-07-03,2026-06-25,2026-09-30,2026-06-25,0


In [20]:
conditions = [
                df['dpd'] <= 0,
                df['dpd'].between(1, 30),
                df['dpd'].between(31, 60),
                df['dpd'].between(61, 90)
            ]

category = ['bucket0', 'bucket1', 'bucket2', 'bucket3']

df['bucket'] = np.select(conditions, category, default='bucket4')

summary = pd.pivot_table(df, index='bucket',aggfunc={'customer_id':'count','outstanding_balance':'sum'}).reset_index()

summary['%age_acct'] = ((summary['customer_id']/summary['customer_id'].sum())*100).round(2)

summary['%age_balance'] = ((summary['outstanding_balance']/summary['outstanding_balance'].sum())*100).round(2)

summary

,bucket,customer_id,outstanding_balance,%age_acct,%age_balance
0,bucket0,53,26169999.46,50.48,48.36
1,bucket1,22,3338554.40,20.95,6.17
2,bucket2,17,15780672.96,16.19,29.16
3,bucket3,8,1501402.20,7.62,2.77
4,bucket4,5,7322000.85,4.76,13.53
